# Pembersihan Data: dataset_mentah_ataraxia.csv menjadi dataset_clean_ataraxia.csv

Notebook ini membersihkan data mentah dan menyimpan hasilnya ke `data/dataset_clean_ataraxia.csv`.

In [ ]:
import pandas as pd

LABELS = ["senang", "sedih", "marah", "takut", "kaget", "jijik", "cape", "netral"]
RAW_PATH = "data/dataset_mentah_ataraxia.csv"
CLEAN_PATH = "data/dataset_clean_ataraxia_hasil.csv"

## 1. Memuat data mentah

In [ ]:
raw = pd.read_csv(RAW_PATH)
print("Jumlah baris mentah:", len(raw))
raw.sample(8, random_state=3)

## 2. Profil masalah pada data mentah

In [ ]:
teks = raw["text"].fillna("")
label = raw["label"].fillna("")
masalah = pd.Series({
    "teks kosong atau hanya spasi": int((teks.str.strip() == "").sum()),
    "label kosong": int((label.str.strip() == "").sum()),
    "label di luar 8 kelas (setelah dirapikan)": int((~label.str.strip().str.lower().isin(LABELS) & (label.str.strip() != "")).sum()),
    "label tidak seragam (huruf besar/spasi)": int((label != label.str.strip().str.lower()).sum()),
    "teks bukan huruf kecil": int((teks != teks.str.lower()).sum()),
    "teks dengan spasi berlebih/tab": int(teks.str.contains(r"^\s|\s$|\s{2,}|\t").sum()),
    "teks mengandung URL": int(teks.str.contains(r"https?://").sum()),
    "teks mengandung mention (@)": int(teks.str.contains(r"(?:^|\s)@\w+").sum()),
    "teks mengandung karakter tak terlihat": int(teks.str.contains("\u200b").sum()),
    "baris duplikat persis": int(raw.duplicated().sum()),
})
masalah

## 3. Proses pembersihan

In [ ]:
log = [("data mentah", len(raw))]
d = raw.copy()

# 1. Buang baris dengan teks atau label kosong
d["text"] = d["text"].fillna("").astype(str)
d["label"] = d["label"].fillna("").astype(str)
d = d[(d["text"].str.strip() != "") & (d["label"].str.strip() != "")]
log.append(("hapus teks/label kosong", len(d)))

# 2. Rapikan label
d["label"] = d["label"].str.strip().str.lower()
d = d[d["label"].isin(LABELS)]
log.append(("hapus label di luar 8 kelas", len(d)))

# 3. Rapikan teks: hapus karakter tak terlihat, URL, mention; huruf kecil; rapikan spasi
d["text"] = (d["text"]
             .str.replace("\u200b", "", regex=False)
             .str.replace(r"https?://\S+", "", regex=True)
             .str.replace(r"(?:^|\s)@\w+", " ", regex=True)
             .str.lower()
             .str.replace(r"\s+", " ", regex=True)
             .str.strip())
d = d[d["text"] != ""]
log.append(("normalisasi teks", len(d)))

# 4. Hapus duplikat (setelah normalisasi)
d = d.drop_duplicates().reset_index(drop=True)
log.append(("hapus duplikat", len(d)))

pd.DataFrame(log, columns=["tahap", "jumlah baris"])

## 4. Validasi hasil

In [ ]:
print("Nilai kosong     :", int(d.isna().sum().sum()))
print("Duplikat         :", int(d.duplicated().sum()))
print("Teks duplikat    :", int(d["text"].duplicated().sum()))
print("Label tidak valid:", int((~d["label"].isin(LABELS)).sum()))
d["label"].value_counts()

## 5. Simpan hasil dan bandingkan dengan dataset bersih

Hasil disimpan ke berkas terpisah agar `dataset_clean_ataraxia.csv` yang asli tidak tertimpa, lalu keduanya dibandingkan.

In [ ]:
d.to_csv(CLEAN_PATH, index=False)

asli = pd.read_csv("data/dataset_clean_ataraxia.csv")
a = set(map(tuple, asli[["label", "text"]].values))
b = set(map(tuple, d[["label", "text"]].values))
print("Baris hasil pembersihan :", len(d))
print("Baris dataset bersih    :", len(asli))
print("Sama persis (sebagai himpunan):", a == b)
print("Hanya ada di hasil      :", len(b - a))
print("Hanya ada di dataset asli:", len(a - b))